In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Moni\uncertainty-aware-rul-prediction")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_split = pd.read_csv(PROCESSED_DIR / "train_split.csv")
val_split = pd.read_csv(PROCESSED_DIR / "validation_split.csv")

print("Training shape:", train_split.shape)
print("Validation shape:", val_split.shape)
print("Training RUL range:", train_split["RUL"].min(), train_split["RUL"].max())
print("Validation RUL range:", val_split["RUL"].min(), val_split["RUL"].max())
print("Missing values:", train_split.isna().sum().sum(),
      val_split.isna().sum().sum())

Training shape: (36420, 27)
Validation shape: (9498, 27)
Training RUL range: 0 356
Validation RUL range: 0 283
Missing values: 0 0


In [2]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Separate target and features
TARGET = "RUL"

X_train = train_split.drop(columns=[TARGET])
y_train = train_split[TARGET]

X_val = val_split.drop(columns=[TARGET])
y_val = val_split[TARGET]

# Exclude engine ID and cycle index from predictive features
# These identify an observation rather than directly measuring sensor condition
exclude_cols = ["unit_nr", "time_cycles"]

X_train = X_train.drop(columns=exclude_cols, errors="ignore")
X_val = X_val.drop(columns=exclude_cols, errors="ignore")

print("Feature count:", X_train.shape[1])
print("Features:", X_train.columns.tolist())

# Baseline 1: Predict the training-set median RUL
dummy_model = DummyRegressor(strategy="median")
dummy_model.fit(X_train, y_train)
dummy_preds = dummy_model.predict(X_val)

# Baseline 2: Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)

# Evaluation
results = pd.DataFrame({
    "Model": ["Median Baseline", "Random Forest"],
    "MAE": [
        mean_absolute_error(y_val, dummy_preds),
        mean_absolute_error(y_val, rf_preds)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, dummy_preds)),
        np.sqrt(mean_squared_error(y_val, rf_preds))
    ]
})

print(results.round(3).to_string(index=False))

Feature count: 26
Features: ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']
          Model    MAE   RMSE
Median Baseline 58.618 70.111
  Random Forest 29.323 37.462


In [3]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prepare features without engine identity or cycle index
exclude_cols = ["RUL", "engine_id", "cycle"]

X_train = train_split.drop(columns=exclude_cols)
y_train = train_split["RUL"]

X_val = val_split.drop(columns=exclude_cols)
y_val = val_split["RUL"]

# NASA asymmetric scoring function (lower is better)
def nasa_score(y_true, y_pred):
    diff = np.asarray(y_pred) - np.asarray(y_true)
    return np.sum(
        np.where(diff < 0, np.expm1(-diff / 13),
                 np.expm1(diff / 10))
    )

# Train comparable models
models = {
    "Median Baseline": DummyRegressor(strategy="median"),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
        "NASA Score": nasa_score(y_val, predictions)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

          Model    MAE   RMSE   NASA Score
Median Baseline 58.618 70.111 1.040652e+08
  Random Forest 30.513 41.187 5.593909e+06


In [4]:
from pathlib import Path

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(RESULTS_DIR / "baseline_results.csv", index=False)

print("Saved:", RESULTS_DIR / "baseline_results.csv")

Saved: C:\Users\Moni\uncertainty-aware-rul-prediction\results\baseline_results.csv
